# Facilities Maintenance & Reliability deployment

This notebook is the guided entry point for deploying the complete synthetic solution into another Microsoft tenant. It keeps the implementation in reusable Python and PowerShell modules while executing them in dependency order.

## What it deploys

1. Synthetic work-management data and controlled documents
2. Fabric workspace, lakehouse, and SQL analytics endpoint
3. Relational Fabric SQL operational replica
4. Delta staging tables and Direct Lake semantic model over Fabric SQL
5. Fabric Data Agent analytics and optional Azure AI Search work-order RAG
6. SharePoint site and controlled-document libraries
7. Copilot Studio SharePoint knowledge and optional Azure AI Search connector

> The notebook updates only your local `.env` with IDs returned by deployment. `.env` and generated deployment outputs are excluded from Git.

## Prerequisites

- Windows PowerShell 7 (`pwsh`) or Windows PowerShell
- Python 3.11 or newer
- Azure CLI authenticated with `az login`
- Access to an active or resumable Microsoft Fabric capacity
- Permission to create Fabric workspace items
- Permission to create Microsoft 365 groups, SharePoint sites, Entra applications, and Graph role assignments
- An Azure OpenAI embedding deployment

Copy `.env.example` to `.env` before continuing. Initially fill in the Fabric capacity, Azure subscription/OpenAI, Microsoft tenant, and SharePoint host values. Resource IDs created by this notebook can remain blank.

In [ ]:
from __future__ import annotations

import json
import os
import shutil
import subprocess
import sys
from pathlib import Path

ROOT = Path.cwd()
if not (ROOT / "README.md").exists():
    raise RuntimeError("Open this notebook with the repository root as the working directory.")

ENV_PATH = ROOT / ".env"
GENERATED = ROOT / "generated"
POWERSHELL = shutil.which("pwsh") or shutil.which("powershell")

def run(command: list[str]) -> None:
    print("$", subprocess.list2cmdline(command))
    subprocess.run(command, cwd=ROOT, check=True)

def read_json(name: str) -> dict:
    return json.loads((GENERATED / name).read_text(encoding="utf-8-sig"))

def read_env() -> dict[str, str]:
    values = {}
    if ENV_PATH.exists():
        for raw_line in ENV_PATH.read_text(encoding="utf-8-sig").splitlines():
            line = raw_line.strip()
            if line and not line.startswith("#") and "=" in line:
                name, value = line.split("=", 1)
                values[name.strip()] = value.strip()
    return values

def update_env(updates: dict[str, object]) -> None:
    lines = ENV_PATH.read_text(encoding="utf-8-sig").splitlines()
    pending = {key: str(value) for key, value in updates.items() if value}
    result = []
    for line in lines:
        if "=" in line and not line.lstrip().startswith("#"):
            name = line.split("=", 1)[0].strip()
            if name in pending:
                line = f"{name}={pending.pop(name)}"
        result.append(line)
    if pending:
        result.extend(f"{name}={value}" for name, value in pending.items())
    ENV_PATH.write_text("\n".join(result) + "\n", encoding="utf-8")

print(f"Repository: {ROOT}")
print(f"PowerShell: {POWERSHELL}")

In [ ]:
# Preflight: fail before provisioning if bootstrap configuration or tools are missing.
if not ENV_PATH.exists():
    raise FileNotFoundError("Copy .env.example to .env and configure it before deployment.")
if not POWERSHELL:
    raise RuntimeError("PowerShell was not found. Install PowerShell 7 or use Windows PowerShell.")
if not shutil.which("az"):
    raise RuntimeError("Azure CLI was not found.")

bootstrap_variables = [
    "FABRIC_WORKSPACE_NAME",
    "FABRIC_CAPACITY_ID",
    "FABRIC_CAPACITY_RESOURCE_ID",
    "FABRIC_LAKEHOUSE_NAME",
    "AZURE_SUBSCRIPTION_ID",
    "MICROSOFT_TENANT_ID",
    "SHAREPOINT_TENANT_HOST",
    "SHAREPOINT_SITE_TITLE",
    "SHAREPOINT_SITE_ALIAS",
]
env = read_env()
work_order_rag_enabled = env.get("ENABLE_WORK_ORDER_RAG", "false").casefold() in {"1", "true", "yes", "on"}
if work_order_rag_enabled:
    bootstrap_variables.extend([
        "AZURE_OPENAI_RESOURCE",
        "AZURE_OPENAI_RESOURCE_GROUP",
        "AZURE_OPENAI_EMBEDDING_DEPLOYMENT",
    ])
missing = [name for name in bootstrap_variables if not env.get(name)]
if missing:
    raise RuntimeError(f"Set these values in .env before deployment: {', '.join(missing)}")

run([sys.executable, "-m", "pip", "install", "-r", "requirements.txt"])
run(["az", "account", "show", "--output", "none"])
print("Preflight passed.")

## 1. Generate deterministic synthetic assets

This creates the Parquet, PDF, and Word source assets under `generated/`. The fixed random seed makes the demo repeatable.

In [ ]:
run([sys.executable, "deployment/generate_demo.py"])
manifest = read_json("manifest.json")
print(json.dumps(manifest, indent=2)[:2000])

## 2. Provision the Fabric workspace and lakehouse

The script is idempotent by display name. It uploads the generated source assets and returns the workspace, lakehouse, and SQL analytics endpoint IDs.

In [ ]:
run([POWERSHELL, "-NoProfile", "-File", "deployment/provision_fabric.ps1"])
fabric = read_json("fabric-deployment.json")
update_env({
    "FABRIC_WORKSPACE_ID": fabric["workspaceId"],
    "FABRIC_LAKEHOUSE_ID": fabric["lakehouseId"],
    "FABRIC_SQL_ENDPOINT_ID": fabric["sqlEndpointId"],
})
print(json.dumps(fabric, indent=2))

## 3. Provision the relational Fabric SQL replica

This creates the SQL serving database, loads the Maximo-shaped replica, and removes legacy SQL vector-search objects from earlier versions of the lab.

In [ ]:
run([sys.executable, "deployment/provision_fabric_sql.py"])
sql_deployment = read_json("fabric-sql-deployment.json")
update_env({
    "FABRIC_SQL_DATABASE_ID": sql_deployment["databaseId"],
    "FABRIC_SQL_CONNECTION_STRING": sql_deployment["connectionString"],
})
print(json.dumps(sql_deployment, indent=2, default=str)[:4000])

## 4. Provision optional analogous-work-order RAG

When `ENABLE_WORK_ORDER_RAG=true`, this creates Azure AI Search, builds a hybrid semantic/vector index over completed work orders, uploads 1,536-dimension embeddings, and runs a five-result retrieval smoke test. Fabric SQL remains relational.

In [ ]:
run([sys.executable, "deployment/provision_work_order_rag.py"])
rag = read_json("work-order-rag-deployment.json")
if rag.get("enabled"):
    update_env({
        "AZURE_AI_SEARCH_RESOURCE_GROUP": rag["resourceGroup"],
        "AZURE_AI_SEARCH_SERVICE": rag["serviceName"],
        "AZURE_AI_SEARCH_INDEX": rag["indexName"],
    })
print(json.dumps(rag, indent=2))

## 5. Materialize Delta staging tables and deploy the Direct Lake semantic model

The semantic model uses Direct Lake over the Fabric SQL operational database. The generated Delta tables remain available in the lakehouse as reproducible staging assets.

In [ ]:
run([sys.executable, "deployment/materialize_tables_and_model.py"])
run([POWERSHELL, "-NoProfile", "-File", "deployment/provision_tables_and_model.ps1"])
model = read_json("table-model-deployment.json")
update_env({"FABRIC_SEMANTIC_MODEL_ID": model["semanticModelId"]})
print(json.dumps(model, indent=2))

## 6. Publish and test the Fabric Data Agent

The provisioning script attaches the semantic model and runs the deterministic analytics smoke test.

In [ ]:
run([sys.executable, "deployment/provision_data_agent.py"])
agent = read_json("data-agent-deployment.json")
update_env({"FABRIC_DATA_AGENT_ID": agent["agentId"]})
print(json.dumps({
    "agentId": agent["agentId"],
    "analyticsQuestion": agent["smokeTestQuestion"],
    "analyticsResponse": agent["smokeTestResponse"],
}, indent=2))

## 7. Provision SharePoint controlled knowledge

The script creates or reuses the Microsoft 365 group and site, creates both document libraries, uploads the generated documents, and removes its temporary Entra provisioning application in a `finally` block.

In [ ]:
run([POWERSHELL, "-NoProfile", "-File", "deployment/provision_sharepoint.ps1"])
sharepoint = read_json("sharepoint-deployment.json")
print(json.dumps({
    "siteTitle": sharepoint["siteTitle"],
    "siteUrl": sharepoint["siteUrl"],
    "libraries": sharepoint["libraries"],
    "uploadedFiles": sharepoint["uploadedFiles"],
}, indent=2))

## 8. Configure Copilot Studio knowledge and work-order retrieval

The script idempotently creates or updates both SharePoint knowledge sources. When `ENABLE_WORK_ORDER_RAG=true`, it also creates or updates the standard Azure AI Search connector tool with invoker authentication, fixed index/vector fields, a completed-status filter, and a five-result limit. It then publishes the agent.

In [ ]:
run([POWERSHELL, "-NoProfile", "-File", "deployment/provision_copilot_knowledge.ps1"])
knowledge = read_json("copilot-knowledge-deployment.json")
print(json.dumps(knowledge, indent=2))

## 9. Complete Copilot Studio configuration

Copilot Studio connection references and pulled `.mcs` metadata are tenant-specific, so they are intentionally not committed.

1. Create or open an agent using the GitHub Copilot harness.
2. Apply the reusable instructions in `copilot-studio/settings.mcs.yml`.
3. Add **Fabric IQ Data MCP** as a tool and select the published Fabric Data Agent shown above.
4. Confirm the notebook-created **Standard Operating Procedures** and **Equipment Technical Manuals** knowledge sources are ready.
5. When work-order RAG is enabled, create the generated Azure AI Search connection using **Access Key** authentication, endpoint `https://<service-name>.search.windows.net`, and an administrator key retrieved during configuration. Power Platform stores the key in its encrypted connection store; never write it to repository files or deployment JSON.
6. Use invoker authentication for Fabric and Azure AI Search, share the Search connection only with authorized agent users, and validate end-user permissions.
7. Run the prompts in `docs/OGE-Facilities-Maintenance-Demo-Script.docx`.

If you use `pac copilot pull`, its environment-specific `.mcs`, connection, and translation outputs remain local because they are ignored by Git.

In [ ]:
# Final deployment summary
summary = {
    "workspace": read_json("fabric-deployment.json").get("workspaceUrl"),
    "semanticModel": read_json("table-model-deployment.json").get("modelUrl"),
    "dataAgentMcp": read_json("data-agent-deployment.json").get("mcpUrl"),
    "sharePoint": read_json("sharepoint-deployment.json").get("siteUrl"),
    "copilotKnowledge": read_json("copilot-knowledge-deployment.json").get("knowledgeSources"),
    "workOrderRag": read_json("copilot-knowledge-deployment.json").get("workOrderRag"),
}
print(json.dumps(summary, indent=2))
print("Deployment automation completed. Verify the Copilot Studio connections above.")